<a href="https://colab.research.google.com/github/kmmmm25/KumaGPT/blob/main/Kuma_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
class Tokenizer:
    def __init__(self, chars: list[str]) -> None:
        self.str_to_idx: dict[str, int] = dict()
        self.str_to_idx['<|endoftext|>'] = 0
        # utf-8
        for i in range(256):
            if f'<utf8_{i}>' not in self.str_to_idx:
                self.str_to_idx[f'<utf8_{i}>'] = len(self.str_to_idx)
        for char in chars:
            self.str_to_idx[char] = len(self.str_to_idx) if char not in self.str_to_idx else self.str_to_idx[char]

        # 登録したIDに重複がないか確認
        assert len(self.str_to_idx.values()) == len(set(self.str_to_idx.values()))

        self.idx_to_str: dict[int, str] = dict()
        for key, value in self.str_to_idx.items():
            self.idx_to_str[value] = key

    def encode(self, text: str, eot=False) -> list[int]:
        result: list[int] = []
        for char in text:
            maybe_token: int | None = self.str_to_idx.get(char)
            if maybe_token is not None:
                result.append(self.str_to_idx[char])
            else:
                utf_8_num: list[int] = list(char.encode("utf-8"))
                for num in utf_8_num:
                    result.append(self.str_to_idx[f'<utf8_{num}>'])
        if eot:
            result.append(self.str_to_idx['<|endoftext|>'])
        return result

    def decode(self, tokens: list[int]) -> str:
        decoded_with_utf_token: list[str] = [self.idx_to_str[token] for token in tokens]
        decoded_postprocess_utf: list[str] = []
        utf_tokens: list[int] = []
        for token in decoded_with_utf_token:
            if token.startswith("<utf8_"):
                utf_num = int(token.replace("<utf8_", "").replace(">", ""))
                utf_tokens.append(utf_num)
            else:
                if utf_tokens:
                    decoded_postprocess_utf.append(bytes(utf_tokens).decode("utf-8", errors="replace"))
                    utf_tokens = []
                decoded_postprocess_utf.append(token)
        if utf_tokens:
            decoded_postprocess_utf.append(bytes(utf_tokens).decode("utf-8", errors="replace"))
            utf_tokens = []
        return "".join(decoded_postprocess_utf)

    def decode_with_utf(self, tokens:list[int]) -> str:
        return "".join([self.idx_to_str[token] for token in tokens])

In [2]:
!pip install -q datasets

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "IVUL-KAUST/MOLE",
    split="train",
    streaming=True
)

README.md:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

In [4]:
dataset2 = load_dataset(
    "wikimedia/wikipedia",
    "20231101.ja",
    split="train",
    streaming=True
)

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

In [5]:
texts = []
count = 0

# Iterate through the first dataset (IVUL-KAUST/MOLE)
for item in dataset:
    if "Description" in item:
        texts.append(item["Description"])
        count += 1
        if count >= 1000:
            break

# If we haven't reached 1000 items yet, iterate through the second dataset (wikimedia/wikipedia)
if count < 1000:
    for item in dataset2:
        if "text" in item:
            texts.append(item["text"])
            count += 1
            if count >= 1000:
                break

all_text = "\n".join(texts)

In [6]:
all_text = "".join(texts)

vocab = sorted(set(all_text))

tokenizer = Tokenizer(vocab)

vocab_size = len(tokenizer.str_to_idx)

print(vocab_size)

5125


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(2005)

class Attention(nn.Module):
  def __init__(self, d_model, d_head, h):
    assert d_head % h == 0

    super().__init__()

    self.Wk = nn.Linear(d_model, d_head)
    self.Wq = nn.Linear(d_model, d_head)
    self.Wv = nn.Linear(d_model, d_head)

    self.h = h

    self.linear = nn.Linear(d_head, d_model)

  def forward(self, x):

    k = self.Wk(x)
    q = self.Wq(x)
    v = self.Wv(x)

    B, T, d_head = k.shape

    k = k.view(B, T, self.h, d_head // self.h).transpose(1, 2)
    q = q.view(B, T, self.h, d_head // self.h).transpose(1, 2)
    v = v.view(B, T, self.h, d_head // self.h).transpose(1, 2)

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    # (B, h, T, d_head)
    #          ↓
    # (B, T, h, d_head)
    y = y.transpose(1, 2)

    attention = y.reshape(B, T, d_head)

    output = self.linear(attention)

    return output

class Block(nn.Module):
  def __init__(self, d_model, d_head, d_ff, h):
    super().__init__()
    self.ln1 = nn.LayerNorm(d_model)
    self.attn = Attention(d_model, d_head, h)

    self.ln2 = nn.LayerNorm(d_model)
    self.ff = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_model)
    )

  def forward(self, x):
    x = x + self.attn(self.ln1(x))

    x = x + self.ff(self.ln2(x))

    return x


class Kuma_GPT(nn.Module):
  def __init__(self, vocab_size, block_size, d_model, d_head, d_ff, n_layer, h):
    super().__init__()
    self.token_embedding = nn.Embedding(vocab_size, d_model)
    self.pos_embedding = nn.Embedding(block_size, d_model)

    self.blocks = nn.ModuleList([
        Block(d_model, d_head, d_ff, h)
        for _ in range(n_layer)
    ])

    self.lnf = nn.LayerNorm(d_model)
    self.linear_output = nn.Linear(d_model, vocab_size)

    self.block_size = block_size

  def forward(self, input, targets=None):
    B, T = input.shape

    tok_emb = self.token_embedding(input)

    pos_input = torch.arange(T, device=input.device)
    pos_emb = self.pos_embedding(pos_input)

    x = tok_emb + pos_emb

    for block in self.blocks:
      x = block(x)

    y = self.lnf(x)
    logits = self.linear_output(y)

    loss = None
    if targets is not None:
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)

    return logits, loss

  @torch.no_grad()
  def generate(self, input, max):
    results = input.clone()

    for _ in range(max):
      x = results[:, -self.block_size:]
      logits, _ = self(x)

      next = logits[:, -1, :]
      probs = F.softmax(next, dim=-1)

      next_token = torch.multinomial(probs, num_samples=1)
      results = torch.cat([results, next_token], dim=1)

    return results

In [14]:
block = 256
d_model = 128
d_ff = d_model * 4
d_head = d_model
n_layer = 4
h = 4

In [17]:
train_texts = texts[:]

print(len(train_texts))

train_tokens = []
for x in train_texts:
  tokens = tokenizer.encode(x)
  train_tokens.append(tokens)

1000


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

flat_tokens = [token for sublist in train_tokens for token in sublist]

batch_size = 32

chunk_size = batch_size + 1

num_chunks = len(flat_tokens) // chunk_size

train_tokens = torch.tensor(flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

epoch = 1

model = Kuma_GPT(vocab_size, block, d_model, d_head, d_ff, n_layer, h).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for i in range(epoch):
    for j in range(0, len(train_tokens), batch_size):

        batch_token = train_tokens[j:j+batch_size].to(device)

        input = batch_token[:, :-1]
        target = batch_token[:, 1:]

        optimizer.zero_grad()

        logits, loss = model(input, target)

        loss.backward()
        optimizer.step()


    torch.save({
      "epoch": epoch,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "loss": loss.item(),
      }, "/content/drive/MyDrive/KumaGPT/checkpoint.pt")

    print(f'epoch: {i}, batch_size: {int(j/batch_size)}/{len(train_tokens)//batch_size + 1}, loss: {loss.item():.6f}')


print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {batch_size}")
print(f"Block size: {block}")
print(f"Tokens / step: {batch_size * block:,}")
print(f"Training steps: {num_chunks:,}")
print(f"Total training tokens: {batch_size * block * num_chunks:,}")
print(f"Total training tokens: {batch_size * block * num_chunks / 1e6:.2f}M")


NameError: name 'torch' is not defined

In [1]:
sentence = "名称"
sentence_token = tokenizer.encode(sentence)

x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

y = model.generate(x, 20)

print(tokenizer.decode(y[0].tolist()))

NameError: name 'tokenizer' is not defined